**Signal Diagnosis**

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system"
)

data_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_churn_features.csv"
)

model_data = pd.read_csv(data_path)

print(model_data.shape)

(10000, 41)


In [ ]:
# Correlation Analysis for numeric Features

numeric_cols = (
    model_data
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

numeric_cols.remove("Churn")

correlations = (
    model_data[numeric_cols + ["Churn"]] 
    .corr(numeric_only=True)["Churn"]
    .drop("Churn")
    .sort_values(
        key=abs,
        ascending=False
    )
)

display(correlations.head(15))

sms_avg                     0.022373
sms_avg_first3m             0.017468
complaints_total            0.016243
complaints_avg              0.016243
MonthlyCharges             -0.010871
data_max                   -0.010202
call_max                    0.009839
complaints_total_first3m    0.009432
complaints_change_3m       -0.009101
call_avg_last3m             0.008561
Age                         0.008433
sms_change_3m              -0.008326
data_std                   -0.007831
call_std                    0.007416
data_avg                   -0.007242
Name: Churn, dtype: float64

In [ ]:
#Correlation Analysis for categorical Features 

categorical_cols = [
    "Gender",
    "Region",
    "ContractType"
]

for col in categorical_cols:
    print(f"\n{col}")

    result = (
        model_data
        .groupby(col)["Churn"]
        .agg(
            churn_rate="mean",
            customers="count"
        )
        .sort_values(
            "churn_rate",
            ascending=False
        )
    )

    result["churn_rate"] = (
        result["churn_rate"] * 100
    ).round(2)

    display(result)


Gender


,churn_rate,customers
Gender,,
Male,8.02,4978
Female,7.71,5022



Region


,churn_rate,customers
Region,,
Urban,8.12,3287
Suburban,8.05,3402
Rural,7.40,3311



ContractType


,churn_rate,customers
ContractType,,
Yearly,8.82,2006
Prepaid,7.67,4041
Monthly,7.56,3953


In [5]:
#Comparison of mean values for important features between churned and non-churned customers

important_features = [
    "MonthlyCharges",
    "TenureMonths",
    "call_avg",
    "data_avg",
    "sms_avg",
    "complaints_total",
    "call_change_3m",
    "data_change_3m",
    "sms_change_3m",
    "complaints_change_3m",
    "call_trend_slope",
    "data_trend_slope",
    "complaints_trend_slope"
]

comparison = (
    model_data
    .groupby("Churn")[important_features]
    .mean()
    .T
)

comparison.columns = [
    "Non_Churn_Mean",
    "Churn_Mean"
]

comparison["Difference"] = (
    comparison["Churn_Mean"]
    - comparison["Non_Churn_Mean"]
)

display(comparison)

,Non_Churn_Mean,Churn_Mean,Difference
MonthlyCharges,51.095953,50.227150,-0.868803
TenureMonths,41.611331,41.606461,-0.004869
call_avg,492.195052,494.827451,2.632400
data_avg,9.900067,9.875262,-0.024805
sms_avg,24.498182,24.844360,0.346178
complaints_total,6.024419,6.171756,0.147336
call_change_3m,-24.782906,-23.046370,1.736537
data_change_3m,-0.418742,-0.415899,0.002843
sms_change_3m,0.026337,-0.339271,-0.365607
complaints_change_3m,-0.012264,-0.071247,-0.058983


**nonlinear signal diagnostic: Mutual Information**

In [6]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [7]:
#Prepare features

X_diag = model_data.drop(
    columns=[
        "CustomerID",
        "SignupDate",
        "FirstObservedUsageMonth",
        "Churn"
    ]
)

y_diag = model_data["Churn"]

categorical_diag = X_diag.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_diag = X_diag.select_dtypes(
    include=["number"]
).columns.tolist()

In [9]:
#Diagnostic preprocessing

diag_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_diag
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_diag
        )
    ]
)

In [ ]:
X_diag_processed = diag_preprocessor.fit_transform(X_diag)


feature_names = (
    numeric_diag
    + diag_preprocessor
      .named_transformers_["cat"]
      .get_feature_names_out(
          categorical_diag
      )
      .tolist()
)

In [14]:
#Calculate mutual information scores for each feature

mi_scores = mutual_info_classif(
    X_diag_processed,
    y_diag,
    random_state=42
)

mi_results = (
    pd.DataFrame({
        "Feature": feature_names,
        "MutualInformation": mi_scores
    })
    .sort_values(
        "MutualInformation",
        ascending=False
    )
)

display(
    mi_results.head(15)
)

,Feature,MutualInformation
30,data_trend_slope,0.005488
13,sms_std,0.004306
3,MonthlyChargesInvalid,0.004257
5,call_std,0.003917
40,ContractType_Prepaid,0.003716
38,Region_Urban,0.003063
28,complaints_change_3m,0.002381
36,Region_Rural,0.002322
34,Gender_Female,0.002247
4,call_avg,0.001903
